# WITNESS-2v2: The Contestants' Rules
**Muse's build, for Kevin · Genesis Seeds · 2026-10-30**

We asked the three models how to improve the bench. They answered (qwen3
included, now that `think: false` frees it). Adopted: input/output examples
in every spec · thinking disabled for all · **every response logged** ·
degenerate outputs (empty/unparseable) score I = 0 · tiebreak P, then G, then
first-seen. Rejected with reason: penalizing non-recursive code — the spec
promises behavior, not style, and the judge keeps that promise.

Fresh unseen problems: `second_largest`, `is_anagram`.

In [1]:
import json, re, signal, urllib.request
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()
OLLAMA = "http://localhost:11434/api/generate"
AGENTS = [("aria-coder:latest", 0.2), ("qwen2.5-coder:7b", 0.7), ("qwen3:8b", 1.0)]
NAMES = {"aria-coder:latest": "ARIA-CODER", "qwen2.5-coder:7b": "QWEN-CODER", "qwen3:8b": "QWEN3"}
LOG = []

def ask(model, spec, temp, tag):
    body = json.dumps({"model": model, "stream": False, "think": False,
        "prompt": f"{spec}\nReply with exactly one ```python block, nothing else.",
        "options": {"num_predict": 300, "temperature": temp}}).encode()
    req = urllib.request.Request(OLLAMA, data=body, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=600) as r:
        resp = json.load(r)["response"]
    LOG.append({"tag": tag, "model": model, "temp": temp, "response": resp})
    return resp

def extract(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S)
    m = re.search(r"```python\s*(.*?)```", text, re.S)
    return (m.group(1) if m else text).strip()

print("v2 wired: think off, examples on, every response logged")


v2 wired: think off, examples on, every response logged


In [2]:
PROBLEMS = {
  "second_largest": (
    "Write a Python function second_largest(nums) that returns the second largest "
    "DISTINCT integer in the list, or None if there are fewer than two distinct values. "
    "Examples: second_largest([1, 2, 3]) == 2; second_largest([5, 5, 5]) is None. "
    "Name it exactly second_largest."),
  "is_anagram": (
    "Write a Python function is_anagram(a, b) that returns True if the two strings are "
    "anagrams of each other, case-insensitive, ignoring spaces. "
    "Examples: is_anagram('Listen', 'Silent') == True; is_anagram('hello', 'world') == False. "
    "Name it exactly is_anagram."),
}
for k, v in PROBLEMS.items():
    print(f"--- {k} ---\n{v}\n")


--- second_largest ---
Write a Python function second_largest(nums) that returns the second largest DISTINCT integer in the list, or None if there are fewer than two distinct values. Examples: second_largest([1, 2, 3]) == 2; second_largest([5, 5, 5]) is None. Name it exactly second_largest.

--- is_anagram ---
Write a Python function is_anagram(a, b) that returns True if the two strings are anagrams of each other, case-insensitive, ignoring spaces. Examples: is_anagram('Listen', 'Silent') == True; is_anagram('hello', 'world') == False. Name it exactly is_anagram.



In [3]:
def ref_second_largest(nums):
    u = sorted(set(nums), reverse=True)
    return u[1] if len(u) >= 2 else None

def ref_is_anagram(a, b):
    norm = lambda s: sorted(s.replace(" ", "").lower())
    return norm(a) == norm(b)

PROBES = {
  "second_largest": [[3, 1, 2], [5, 5, 5], [], [7], [-1, -5, -3], [10, 10, 9, 9, 8], [0, -1], [2, 2, 1]],
  "is_anagram": [("Listen", "Silent"), ("hello", "world"), ("", ""), ("a", "A"),
                 ("dormitory", "dirty room"), ("abc", "abcd"),
                 ("Tom Marvolo Riddle", "I am Lord Voldemort"), ("a b", "ab ")],
}
REFS = {"second_largest": ref_second_largest, "is_anagram": ref_is_anagram}
TRUTH = {p: [REFS[p](*(x if isinstance(x, tuple) else [x])) for x in PROBES[p]] for p in PROBES}
DEGEN = {"NO-FUNC", "COMPILE-ERR", "TIMEOUT"}

class Timeout(Exception):
    pass

def _alarm(signum, frame):
    raise Timeout()

def run_candidate(code, fname, probes):
    ns = {}
    try:
        exec(compile(code, "<candidate>", "exec"), {"__builtins__": __builtins__}, ns)
        fn = ns.get(fname)
        if not callable(fn):
            return ["NO-FUNC"] * len(probes)
    except Exception:
        return ["COMPILE-ERR"] * len(probes)
    out = []
    for x in probes:
        args = x if isinstance(x, tuple) else (x,)
        signal.signal(signal.SIGALRM, _alarm)
        signal.alarm(3)
        try:
            out.append(fn(*args))
        except Timeout:
            out.append("TIMEOUT")
        except Exception:
            out.append("ERR")
        finally:
            signal.alarm(0)
    return out

def handshake_sample(coup, shots=400):
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.crx(coup, 0, 1)
    qc.measure([0, 1], [0, 1])
    counts = sim.run(transpile(qc, sim), shots=shots).result().get_counts()
    return (counts.get("00", 0) + counts.get("11", 0)) / shots

def behavior_agree(out1, out2):
    return sum(a == b for a, b in zip(out1, out2)) / len(out1)

def valid(vec):
    return not any(v in DEGEN for v in vec)

print("v2 harness ready:", {p: len(v) for p, v in PROBES.items()}, "probes each")


v2 harness ready: {'second_largest': 8, 'is_anagram': 8} probes each


In [4]:
import time
REASKS = 2
BAR = 0.60
spent_calls = 0

def timed_ask(model, spec, temp, tag):
    global spent_calls
    t0 = time.time()
    out = ask(model, spec, temp, tag)
    spent_calls += 1
    print(f"    ask #{spent_calls}: {NAMES[model]} t={temp} ({time.time()-t0:.0f}s)")
    return out

for pname, spec in PROBLEMS.items():
    print(f"\n===== {pname} =====")
    truth = TRUTH[pname]
    samples = {}
    for model, temp in AGENTS:
        samples[model] = []
        for k, dt in enumerate((0.0, 0.2)):
            resp = timed_ask(model, spec, round(temp + dt, 1), f"{pname}/{NAMES[model]}#{k}")
            samples[model].append(run_candidate(extract(resp), pname, PROBES[pname]))
    models = [m for m, _ in AGENTS]
    coup = {}
    for i in range(3):
        for j in range(i + 1, 3):
            a = behavior_agree(samples[models[i]][0], samples[models[j]][0])
            coup[(i, j)] = a * np.pi
            print(f"  {NAMES[models[i]]}~{NAMES[models[j]]}: raw agree {a:.2f} "
                  f"-> measured {handshake_sample(coup[(i, j)]):.2f}")
    mean_agree = {}
    for i, m in enumerate(models):
        others = [handshake_sample(coup[tuple(sorted((i, j)))]) for j in range(3) if j != i]
        mean_agree[m] = sum(others) / len(others)
    weak = min(models, key=lambda m: mean_agree[m])
    print(f"  witness spends {REASKS} re-asks on {NAMES[weak]} (mean measured {mean_agree[weak]:.2f})")
    for k in range(REASKS):
        resp = timed_ask(weak, spec, 1.2, f"{pname}/{NAMES[weak]}#re{k}")
        samples[weak].append(run_candidate(extract(resp), pname, PROBES[pname]))
    print(f"  --- scorecard (I gated: <2 valid samples -> I = 0) ---")
    Q, Ptab, Gtab = {}, {}, {}
    for m in models:
        vecs = samples[m]
        P = sum(1 for v, t in zip(vecs[0], truth) if v == t) / len(truth)
        E3 = 0.0 if not valid(vecs[0]) else min(1.0,
            (sum(1 for v, t in zip(vecs[1], truth) if v == t) / len(truth)) / max(P, 1e-9))
        vv = [v for v in vecs if valid(v)]
        I = sum(behavior_agree(vv[0], v) for v in vv[1:]) / max(len(vv) - 1, 1) if len(vv) >= 2 else 0.0
        G = max(sum(1 for v, t in zip(v, truth) if v == t) / len(truth) for v in vecs)
        Q[m] = (P + E3 + I + G) / 4.0
        Ptab[m], Gtab[m] = P, G
        print(f"  {NAMES[m]}: P {P:.2f} E3 {E3:.2f} I {I:.2f} G {G:.2f} -> Q {Q[m]:.3f}")
    ranked = sorted(models, key=lambda m: (Q[m], Ptab[m], Gtab[m]), reverse=True)
    best = ranked[0]
    if Q[best] < BAR:
        print(f"  REFUSED: best Q {Q[best]:.3f} below bar {BAR}.")
    else:
        bv = max(samples[best], key=lambda v: sum(1 for x, t in zip(v, truth) if x == t))
        print(f"  CROWNED: {NAMES[best]} Q {Q[best]:.3f} "
              f"({sum(1 for x, t in zip(bv, truth) if x == t)}/{len(truth)} probes)")
print(f"\ntotal ollama calls spent: {spent_calls}")
json.dump(LOG, open("/home/kmon/Genesis-Seeds-new/bench_v2_responses.json", "w"), indent=1)
print(f"responses logged: {len(LOG)}")



===== second_largest =====


    ask #1: ARIA-CODER t=0.2 (7s)


    ask #2: ARIA-CODER t=0.4 (2s)


    ask #3: QWEN-CODER t=0.7 (6s)


    ask #4: QWEN-CODER t=0.9 (2s)


...

    ask #14: QWEN3 t=1.2 (1s)
  ARIA-CODER~QWEN-CODER: raw agree 1.00 -> measured 1.00
  ARIA-CODER~QWEN3: raw agree 1.00 -> measured 1.00


  QWEN-CODER~QWEN3: raw agree 1.00 -> measured 1.00


  witness spends 2 re-asks on ARIA-CODER (mean measured 1.00)


    ask #15: ARIA-CODER t=1.2 (7s)


    ask #16: ARIA-CODER t=1.2 (1s)
  --- scorecard (I gated: <2 valid samples -> I = 0) ---
  ARIA-CODER: P 1.00 E3 1.00 I 1.00 G 1.00 -> Q 1.000
  QWEN-CODER: P 1.00 E3 1.00 I 1.00 G 1.00 -> Q 1.000
  QWEN3: P 1.00 E3 1.00 I 1.00 G 1.00 -> Q 1.000
  CROWNED: ARIA-CODER Q 1.000 (8/8 probes)

total ollama calls spent: 16
responses logged: 16


## Discussion — written after the run

**A perfect scoreboard that indicts its own exam: 1.000 everywhere, both
problems, all three models.** Read it right:

- **qwen3 revived (0.25 → 1.000):** the `think: false` fix is confirmed. The v1
  zero was rig, not reasoning — now proven, not assumed.
- **The I-gate never fired** (no degenerate outputs) and **16/16 responses
  logged** — the contestants' harness rules held with nothing to catch.
- **Ceiling effect:** the problems were too easy to separate anyone. Both
  crowns are 3-way ties broken by first-seen — honest but arbitrary. v3 needs
  harder held-out specs (and the bench needs a difficulty ladder, not fixed bars).
- **The rejected rule stands:** nobody needed a recursion style-penalty;
  behavior sufficed. The judge's overrule holds.

Verdict: v2 proved the *rules*, not the ranking. The models improved their own
game — now the game must get harder.


In [5]:
import sys
import qiskit, qiskit_aer
print("python", sys.version.split()[0])
print("qiskit", qiskit.__version__, "| aer", qiskit_aer.__version__)


python 3.14.0
qiskit 2.6.1 | aer 0.18.2
